Note: The script uses Berkeley Neural Parser to parse the generated instructions, and visualize the results using Plotly.

Please make sure to install benepar following their documentation [here](https://github.com/nikitakit/self-attentive-parser#installation).

In [1]:
import benepar, spacy
nlp = spacy.load('en_core_web_md')
doc = nlp("The time for action is now. It's never too late to do something.")

if spacy.__version__.startswith('2'):
    nlp.add_pipe(benepar.BeneparComponent("benepar_en3"))
else:
    nlp.add_pipe("benepar", config={"model": "benepar_en3"})

c:\Users\1J1870897\Anaconda3\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_md' (3.6.0) was trained with spaCy v3.6.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
c:\Users\1J1870897\Anaconda3\lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\1J1870897\Anaconda3\lib\site-packages\pydantic\_internal\_config.py:321: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [2]:
def find_root_verb_and_its_dobj(tree_root):
    # first check if the current node and its children satisfy the condition
    if tree_root.pos_ == "VERB":
        for child in tree_root.children:
            if child.dep_ == "dobj" and child.pos_ == "NOUN":
                return tree_root.lemma_, child.lemma_
        return tree_root.lemma_, None
    # if not, check its children
    for child in tree_root.children:
        return find_root_verb_and_its_dobj(child)
    # if no children satisfy the condition, return None
    return None, None

def find_root_verb_and_its_dobj_in_string(s):
    doc = nlp(s)
    first_sent = list(doc.sents)[0]
    return find_root_verb_and_its_dobj(first_sent.root)

find_root_verb_and_its_dobj_in_string("Write me a story about education.")

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


c:\Users\1J1870897\Anaconda3\lib\site-packages\torch\distributions\distribution.py:45: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(f'{self.__class__} does not define `arg_constraints`. ' +


('write', 'story')

In [3]:
import pandas as pd
import json
import tqdm

df = pd.read_csv('../test/genai_questions_809156896092057937.csv')
df = pd.read_csv('../test/genai_questions_311016922297059474.csv')
instructions = df['questions'].to_list()

raw_phrases = []
for instruction in tqdm.tqdm(instructions):
    try:
        verb, noun = find_root_verb_and_its_dobj_in_string(instruction)
        raw_phrases.append({
            "verb": verb,
            "noun": noun,
            "instruction": instruction
        })
    except Exception as e:
        print(e)
        print(instruction)

  0%|          | 0/407 [00:00<?, ?it/s]

100%|██████████| 407/407 [02:17<00:00,  2.97it/s]


In [4]:
len(raw_phrases)

407

In [5]:
raw_phrases = pd.DataFrame(raw_phrases)
phrases = pd.DataFrame(raw_phrases).dropna()
phrases[["verb", "noun"]].groupby(["verb", "noun"]).size().sort_values(ascending=False)

verb         noun         
provide      information      60
             example          43
affect       performance      11
discuss      impact            4
affect       system            4
discuss      importance        4
affect       operation         4
provide      detail            4
handle       datum             3
discuss      role              3
             off               3
provide      overview          3
affect       reliability       3
             likelihood        3
notice       difference        2
explain      type              2
address      failure           2
affect       component         2
describe     condition         2
affect       datum             2
             lifespan          2
prioritize   task              2
discuss      history           2
             challenge         2
affect       relationship      2
describe     startup           2
explain      role              2
impact       profitability     1
incorporate  knowledge         1
monitor      con

In [6]:
top_verbs = phrases[["verb"]].groupby(["verb"]).size().nlargest(20).reset_index()

df = phrases[phrases["verb"].isin(top_verbs["verb"].tolist())]
# df = df[~df["noun"].isin(["I", "what"])]
# df = phrases
# df[~df["verb"].isin(top_verbs["verb"].tolist())]["verb"] = "other"
# df[~df["verb"].isin(top_verbs["verb"].tolist())]["noun"] = "other"
df = df.groupby(["verb", "noun"]).size().reset_index().rename(columns={0: "count"}).sort_values(by=["count"], ascending=False)
# df = df[df["count"] > 10]
df = df.groupby("verb").apply(lambda x: x.sort_values("count", ascending=False).head(4)).reset_index(drop=True)
df

,verb,noun,count
0,address,failure,2
1,address,challenge,1
2,affect,performance,11
3,affect,system,4
4,affect,operation,4
5,affect,reliability,3
6,bear,failure,1
7,describe,condition,2
8,describe,startup,2
9,detect,issue,1


In [7]:

import plotly.graph_objects as go
import plotly.express as px

# df["blank"] = "ROOT"
# df = phrases.groupby(["verb", "noun"]).size().sort_values(ascending=False).head(5).reset_index().rename(columns={0: "count"})

df = df[df["count"] >= 2]
fig = px.sunburst(df, path=['verb', 'noun'], values='count')
# fig.update_layout(uniformtext=dict(minsize=10, mode='hide'))
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    font_family="Times New Roman",
)
fig.show()
fig.write_html("verb_noun.html")
#fig.savefig("output/verb_noun.pdf")

In [8]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Assuming df and phrases are defined elsewhere in your code
# df = df[df["count"] >= 1]

fig = px.sunburst(df, path=['verb', 'noun'], values='count')
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    font_family="Times New Roman",
)

# Save the plot as a .png file
pio.write_image(fig, 'verb_noun.png')

# Show the plot
fig.show()


In [14]:
#!pip install -U kaleido

     -------------------------------------- 65.9/65.9 MB 743.1 kB/s eta 0:00:00


In [9]:
df['count'].sum()

35